In [1]:
from src.document_loader import load_document

Documents = load_document("data/Employee_Benefits.docx")

print(Documents)

[Document(metadata={'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\enterprise-agentic-rag\\data\\Employee_Benefits.docx'}, page_content='Employee Benefits\n\nEmployees are provided with a comprehensive benefits package designed to support their health, financial security, professional development, and overall well-being. The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations, emergency care, and other eligible healthcare services. In addition to health protection, employees are covered by a life insurance policy that provides financial security for their loved ones, with coverage valued at three times the employee’s annual salary, ensuring meaningful support in the event of an unforeseen circumstance. To encourage continuous learning and career advancement, the organization offers an Education Assistance 

In [2]:
from src.chunking import split_documents

chunks = split_documents(Documents, chunk_size=400, chunk_overlap=50)

print(chunks)

[Document(metadata={'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\enterprise-agentic-rag\\data\\Employee_Benefits.docx', 'chunk_index': 0}, page_content='Employee Benefits'), Document(metadata={'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\enterprise-agentic-rag\\data\\Employee_Benefits.docx', 'chunk_index': 1}, page_content='Employees are provided with a comprehensive benefits package designed to support their health, financial security, professional development, and overall well-being. The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations,'), Document(metadata={'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\enterprise-agentic-rag\\data\\Employee_Benefits.docx', 'chunk_index': 2}, page_content='medical treatments, specialist consultations, emergency care, and other eligible healthcar

In [3]:
import json
from src.retrieval.embedding import get_embedding_model, VectorStoreManager
from langchain_core.documents import Document

with open("config/config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

print(config["llm_provider"])
print(config["embedding_provider"])
print(config["embedding_model"])

embedding_model = get_embedding_model("openai",config["embedding_model"])
print(embedding_model)

from langchain_community.vectorstores import FAISS, Chroma

manager = VectorStoreManager(config=config)
len(chunks)
vector_store = manager.create_index(chunks)

openai
openai
text-embedding-3-small
Loading OpenAI embedding model...
client=<openai.resources.embeddings.Embeddings object at 0x000001E4B7C23890> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001E4B67F2810> model='text-embedding-3-small' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True
Loading OpenAI embedding model...


In [4]:
results = vector_store.similarity_search(
    "What health insurance coverage is available?",
    k=3
)

for r in results:
    print(r.page_content)
    print(r.metadata)

Employees are provided with a comprehensive benefits package designed to support their health, financial security, professional development, and overall well-being. The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations,
{'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\enterprise-agentic-rag\\data\\Employee_Benefits.docx', 'chunk_index': 1}
medical treatments, specialist consultations, emergency care, and other eligible healthcare services. In addition to health protection, employees are covered by a life insurance policy that provides financial security for their loved ones, with coverage valued at three times the employee’s annual salary, ensuring meaningful support in the event of an unforeseen circumstance. To encourage
{'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\enterprise-agentic-rag\\da

In [5]:
results = vector_store.similarity_search_with_score(
    "health insurance",
    k=3
)

for doc, score in results:
    print(f"Score: {score}")
    print(doc.page_content)

Score: 0.9438968896865845
medical treatments, specialist consultations, emergency care, and other eligible healthcare services. In addition to health protection, employees are covered by a life insurance policy that provides financial security for their loved ones, with coverage valued at three times the employee’s annual salary, ensuring meaningful support in the event of an unforeseen circumstance. To encourage
Score: 0.9760313630104065
Employees are provided with a comprehensive benefits package designed to support their health, financial security, professional development, and overall well-being. The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations,
Score: 1.0813671350479126
Employee Benefits
